<a href="https://colab.research.google.com/github/dcruzcavalieri/agentes-2026-2-modelo/blob/main/enc02_primeira_chamada.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Encontro 2 — Sua primeira chamada, e o ambiente que vamos usar o semestre todo

Tópicos Especiais em IA — Agentes Inteligentes · IFES Serra · 2026/2

---

## O que você entrega ao final desta aula

1. Este notebook rodando de ponta a ponta, **salvo no repositório da sua equipe**
2. O consumo de tokens da sua primeira chamada, anotado
3. A mesma pergunta respondida por **dois modelos diferentes**, sem alterar o código

## Antes de começar

No painel esquerdo, ícone de chave (**Secrets**), confirme que existem e estão liberados para este notebook:

- `GROQ_API_KEY` — provedor principal da disciplina
- `GEMINI_API_KEY` — usaremos só no Encontro 9, mas já deixamos pronto

Se faltar alguma, avise agora: os dez primeiros minutos da aula são plantão.

## Duas trilhas

Cada laboratório tem **caminho mínimo** e **caminho estendido**. O mínimo é obrigatório e tem o código quase pronto — você preenche o que está marcado com `# SEU CÓDIGO`. O estendido é para quem terminar antes; não vale nota, vale aprendizado.

## Parte 0 — Célula de preparo

**Toda aula começa por aqui.** A máquina virtual do Colab é apagada entre sessões: o que você instalou ontem não está mais aqui. Por isso todo notebook precisa se reconstruir sozinho.

Hoje instalamos **uma biblioteca só**: o cliente da OpenAI. E vamos usá-lo para conversar com um provedor que **não** é a OpenAI — o que é justamente o ponto da Parte 3.

A versão está limitada com `<3`, e não é preciosismo: bibliotecas desta área quebram a interface entre versões maiores. No Encontro 10, quando entrar um *framework* de agentes, você vai ver a versão fixada com `==` exatamente por isso.

In [1]:
%pip install -q "openai>=1.99.0,<3"

import importlib.metadata as md
print("openai", md.version("openai"))

openai 2.45.0


## Parte 1 — A chave nunca entra no código

Chave de API é como senha. Se ela for para o repositório, qualquer pessoa com acesso pode gastar a sua cota — e apagar o arquivo depois não resolve, porque o histórico guarda tudo.

No Colab, a chave vive nos **Secrets**: pertence à sua conta, você libera por notebook, e ela **não** vai junto quando você compartilha o notebook.

A função abaixo também funciona fora do Colab, lendo variável de ambiente. É um exemplo pequeno de código que roda em dois contextos — ideia que volta várias vezes no semestre.

In [2]:
import os

def obter_chave(nome: str) -> str:
    """Le um segredo dos Secrets do Colab; fora do Colab, da variavel de ambiente."""
    try:
        from google.colab import userdata
        return userdata.get(nome)
    except ImportError:
        valor = os.getenv(nome)
        if not valor:
            raise RuntimeError(f"Defina {nome} nos Secrets do Colab ou no ambiente.")
        return valor

GROQ_KEY = obter_chave("GROQ_API_KEY")

# Confirma que a chave chegou, sem imprimi-la. Nunca imprima uma chave inteira.
print("chave carregada, termina em:", GROQ_KEY[-4:])

chave carregada, termina em: 1uUo


## Parte 2 — Python que você vai usar o semestre todo

Quatro coisas, e não mais que isso. Se você já programa em Python, pule para a Parte 3.

| Construção | Onde aparece nesta disciplina |
|---|---|
| **função com anotação de tipo** | toda ferramenta do agente é uma função anotada |
| ***docstring*** | é o texto que o modelo lê para decidir se usa a ferramenta |
| **dicionário** | mensagens, parâmetros e respostas de API são dicionários |
| **JSON** | o formato em que tudo isso viaja pela rede |

A *docstring* merece atenção: no Encontro 5 você vai descobrir que ela **não é comentário** — é a descrição que o modelo lê para escolher a ferramenta. Uma *docstring* ruim é uma ferramenta que o agente nunca chama.

In [3]:
import json

# Nada nesta celula vai para o modelo. Aqui so ensaiamos as quatro construcoes
# que voce vai usar o semestre todo. A primeira chamada de verdade e na Parte 3.

def temperatura_reator(reator: str) -> str:
    """Le a temperatura atual de um reator da planta.

    Args:
        reator: identificador do reator, por exemplo R-101
    """
    leituras = {"R-101": 87.4, "R-102": 91.2, "R-103": 78.9}
    if reator not in leituras:
        return f"reator {reator} desconhecido"
    return f"{reator}: {leituras[reator]} graus Celsius"


# 1) A funcao anotada, chamada por voce, do jeito normal.
print(temperatura_reator("R-101"))
print(temperatura_reator("R-999"))

# 2) A docstring nao e comentario: e dado, legivel em tempo de execucao.
#    No Encontro 5 e este texto que viaja junto para o modelo, e e por ele
#    que o modelo decide se esta ferramenta serve para a pergunta que recebeu.
print("\n--- o que o modelo vai ler sobre esta ferramenta ---")
print(temperatura_reator.__doc__)

# 3) Dicionario e JSON: o corpo exato da requisicao que a Parte 3 vai enviar.
requisicao = {
    "model": "llama-3.1-8b-instant",
    "messages": [
        {"role": "system", "content": "Você responde a engenheiros. Seja preciso e breve."},
        {"role": "user", "content": "Qual a temperatura do reator R-101?"},
    ],
}
print("\n--- o que sai da sua maquina pela rede ---")
print(json.dumps(requisicao, ensure_ascii=False, indent=2))

R-101: 87.4 graus Celsius
reator R-999 desconhecido

--- o que o modelo vai ler sobre esta ferramenta ---
Le a temperatura atual de um reator da planta.

    Args:
        reator: identificador do reator, por exemplo R-101
    

--- o que sai da sua maquina pela rede ---
{
  "model": "llama-3.1-8b-instant",
  "messages": [
    {
      "role": "system",
      "content": "Você responde a engenheiros. Seja preciso e breve."
    },
    {
      "role": "user",
      "content": "Qual a temperatura do reator R-101?"
    }
  ]
}


## Parte 3 — Sua primeira chamada

### Antes do código: três coisas diferentes, e nenhuma é um modelo da OpenAI

A célula abaixo importa uma biblioteca chamada `openai` e conversa com o **Groq**. Isso costuma causar estranheza, e a estranheza é útil — ela vem de três coisas que normalmente se confundem numa só.

| Camada | O que é | De quem é |
|---|---|---|
| **a biblioteca** `openai` | código Python que roda no **seu** Colab: monta o JSON, faz a requisição HTTP, lê a resposta. **Não contém modelo nenhum.** | OpenAI |
| **o protocolo** | o formato do JSON — `messages`, `role`, `content`, `tools`, `choices`, `usage` | a OpenAI definiu, e virou padrão de fato |
| **o modelo** | `llama-3.1-8b-instant`: os pesos que de fato produzem a resposta | **Meta**, executando em hardware do **Groq** |

A biblioteca leva o nome de quem inventou o formato, não de quem responde. **O modelo é da Meta, servido pelo Groq. A OpenAI não participa desta chamada.**

### A analogia

É o mesmo que acontece com **Modbus**. O protocolo foi criado pela Modicon, mas você usa uma biblioteca Modbus para conversar com um CLP Siemens, um inversor WEG ou um relé Schneider. A biblioteca e o protocolo carregam o nome de quem os criou; o equipamento do outro lado é de quem você quiser.

Aqui, `openai` é a biblioteca Modbus. O Groq é o CLP.

### Por que Groq, e não a OpenAI direto

- **A API da OpenAI não tem camada gratuita** — exige crédito pré-pago. Numa disciplina, isso excluiria quem não tem cartão internacional.
- O Groq oferece conta sem cartão, com limite diário generoso no modelo pequeno, latência muito baixa, e **não retém dados de inferência por padrão** — a melhor política de dados entre os provedores gratuitos avaliados.
- **O Groq não fabrica modelos.** Ele faz hardware e serviço de inferência, e hospeda modelos abertos de terceiros. Você vai ver isso na célula seguinte.

### Por que este modelo

O `llama-3.1-8b-instant` tem o maior limite diário de requisições — o que importa porque um agente multiplica chamadas — e a menor latência. O `llama-3.3-70b-versatile` responde melhor, e você vai compará-los na Parte 6.

> **Groq não é Grok.** O Groq é esta empresa de inferência. O Grok é um modelo da xAI. Os nomes colidem e a confusão é comum.

### O que fica

**Biblioteca, protocolo, modelo e infraestrutura são quatro escolhas independentes.** Repare que no código abaixo o provedor aparece em **três variáveis, e em nenhum outro lugar**. É isso que a Parte 6 vai explorar.

In [4]:
from openai import OpenAI

# --- as tres linhas que definem o provedor ---
LLM_BASE_URL = "https://api.groq.com/openai/v1"
LLM_MODEL = "llama-3.1-8b-instant"
LLM_API_KEY = GROQ_KEY
# ---------------------------------------------

cliente = OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY)

resposta = cliente.chat.completions.create(
    model=LLM_MODEL,
    messages=[
        {"role": "system", "content": "Você responde a engenheiros. Seja preciso e breve."},
        {"role": "user", "content": "Em uma frase: o que diferencia um agente de um chatbot?"},
    ],
)

print(resposta.choices[0].message.content)

Um agente é umsoftware que executa uma tarefa específica com base em regras pré-definidas, enquanto um chatbot é um agente especializado em conversação automática, usando inteligência artificial para entender e responder a perguntas e tarefas do usuário em tempo real.


### Quem mora no Groq — veja você mesmo

A célula abaixo lista os modelos que este provedor serve. Repare nos **fabricantes** que aparecem nos nomes: Meta, Alibaba, OpenAI, e outros.

Se houver algum `openai/gpt-oss-...` na lista, olhe com atenção: **é um modelo feito pela OpenAI, servido pelo Groq.** Ou seja, você poderia usar um modelo da OpenAI nesta disciplina — e ainda assim não estar usando a API da OpenAI. Fabricante do modelo e provedor de inferência são escolhas separadas.

In [5]:
modelos = sorted(m.id for m in cliente.models.list())
print(f"{len(modelos)} modelos disponiveis neste provedor:\n")
for m in modelos:
    print("  ", m)

15 modelos disponiveis neste provedor:

   allam-2-7b
   canopylabs/orpheus-arabic-saudi
   canopylabs/orpheus-v1-english
   groq/compound
   groq/compound-mini
   llama-3.1-8b-instant
   llama-3.3-70b-versatile
   meta-llama/llama-prompt-guard-2-22m
   meta-llama/llama-prompt-guard-2-86m
   openai/gpt-oss-120b
   openai/gpt-oss-20b
   openai/gpt-oss-safeguard-20b
   qwen/qwen3.6-27b
   whisper-large-v3
   whisper-large-v3-turbo


### Leia o consumo — é a conta que você vai pagar

Todo retorno traz quantos tokens foram gastos. **Anote esse número.** No Encontro 14 você vai comparar com o consumo de um agente que raciocina em vários passos, e a diferença entre os dois é o argumento daquela aula.

In [6]:
u = resposta.usage
print(f"entrada : {u.prompt_tokens} tokens")
print(f"saida   : {u.completion_tokens} tokens")
print(f"total   : {u.total_tokens} tokens")

TOKENS_PERGUNTA_SIMPLES = u.total_tokens
print("\nGuarde este numero para comparar no Encontro 14.")

entrada : 66 tokens
saida   : 63 tokens
total   : 129 tokens

Guarde este numero para comparar no Encontro 14.


## Parte 4 — Caminho mínimo: faça você

**Este é o entregável do laboratório.** Escreva uma função que faça uma pergunta ao modelo e devolva a resposta e o total de tokens. Preencha onde está marcado.

Se travar, o código da Parte 3 tem tudo o que você precisa — é copiar e adaptar.

In [7]:
def perguntar(pergunta: str, instrucao: str = "Você responde a engenheiros. Seja breve."):
    """Envia uma pergunta ao modelo e devolve (texto_da_resposta, total_de_tokens)."""
    r = cliente.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": instrucao},
            {"role": "user", "content": pergunta}
            # SEU CÓDIGO: acrescente a mensagem do usuario, com a pergunta recebida
        ],
    )
    texto = None    # SEU CÓDIGO: extraia o texto da resposta
    tokens = None   # SEU CÓDIGO: extraia o total de tokens

    texto = r.choices[0].message.content
    tokens = r.usage.total_tokens

    return texto, tokens


# Teste: as tres perguntas devem responder, e nenhuma deve imprimir None
for p in ("O que é PEAS?",
          "Cite um risco de usar agentes em malha de controle em tempo real.",
          "O modelo executa a ferramenta, ou apenas pede que ela seja executada?"):
    texto, tokens = perguntar(p)
    print(f"[{tokens} tokens] {p}\n  -> {texto}\n")

[129 tokens] O que é PEAS?
  -> PEAS é uma sigla que representa os parâmetros básicos de controle em processos de automação.

P - Processo (ou Processo Real)
E - Erro (diferença entre o valor desejado e o valor atual)
A - Ação (ou Ação de controle)
S - Saída (ou Comando de controle)

[166 tokens] Cite um risco de usar agentes em malha de controle em tempo real.
  -> Um risco de usar agentes em malha de controle em tempo real é a possibilidade de falha de tempo real, ou seja, a malha de controle não reage a tempo nas alterações no sistema, o que pode levar a problemas de segurança e estabilidade. Isso ocorre porque os agentes podem não estar preparados para lidar com cenários que não foram previstos, levando a um tempo de resposta lento ou inexistente.

[156 tokens] O modelo executa a ferramenta, ou apenas pede que ela seja executada?
  -> Em geral, o modelo (ou inteligência artificial) não executa a ferramenta diretamente. Em vez disso, o modelo analisa as informações fornecidas e gere

## Parte 5 — A temperatura, e por que nada disso é determinístico

O modelo sorteia o próximo token de uma distribuição de probabilidade. A `temperature` controla o quanto esse sorteio se espalha: perto de zero ele fica repetitivo e previsível; alto, criativo e instável.

Rode a célula e **compare as saídas**. Este é o problema central do Ciclo 3: como avaliar um sistema que não repete a si mesmo.

In [9]:
PERGUNTA = "Dê um nome curto para um agente que diagnostica falhas em uma planta industrial."

for temp in (0.0, 0.0, 1.2, 1.2):
    r = cliente.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": PERGUNTA}],
        temperature=temp,
        max_tokens=100,
    )
    print(f"temperatura {temp}: {r.choices[0].message.content.strip()}\n")

print("\nAs duas primeiras linhas tendem a coincidir; as duas ultimas, nao.")
print("Guarde a pergunta: como se testa um programa que nao repete a si mesmo?")

temperatura 0.0: Um nome curto para um agente que diagnostica falhas em uma planta industrial pode ser:

- "FaultFinder" (encontrador de falhas)
- "Diagno" (diagnóstico)
- "FaultScan" (escaneamento de falhas)
- "PlantaGuard" (guardião da planta)
- "FaultFix" (correção de falhas)

Escolha o que melhor se adequa ao seu contexto.

temperatura 0.0: Um nome curto para um agente que diagnostica falhas em uma planta industrial pode ser:

- "FaultFinder" (encontrador de falhas)
- "Diagno" (diagnóstico)
- "FaultScan" (escaneamento de falhas)
- "PlantaGuard" (guardião da planta)
- "FaultFix" (correção de falhas)

Escolha o que melhor se adequa ao seu contexto.

temperatura 1.2: Poderíamos sugerir os seguintes nomes curtos para um agente que diagnostica falhas em uma planta industrial:

1. Diagro
2. Fail
3. Systa
4. Alert (um nome comumente utilizado como sistema de alerta em indústrias)
5. Checko

Esse agente seria uma ferramenta importante para identificar e corrigir problemas em tempo real, ga

## Parte 6 — Trocar de modelo sem tocar no código

O exercício central da aula, e a consequência prática da separação em camadas da Parte 3. Abaixo, duas configurações: um modelo pequeno e um grande. O código que faz a pergunta é **idêntico** nos dois casos.

Compare três coisas: a **qualidade** da resposta, o **consumo de tokens** e o **tempo**. Essa escolha de trade-off volta no Encontro 14.

In [10]:
import time

CONFIGURACOES = {
    "pequeno": dict(base_url="https://api.groq.com/openai/v1",
                    model="llama-3.1-8b-instant", key=GROQ_KEY),
    "grande":  dict(base_url="https://api.groq.com/openai/v1",
                    model="llama-3.3-70b-versatile", key=GROQ_KEY),
}

PERGUNTA = ("Um operador relata vibração anormal na bomba P-204. "
            "Liste no máximo três hipóteses de causa, em ordem de probabilidade.")

for nome, cfg in CONFIGURACOES.items():
    try:
        c = OpenAI(base_url=cfg["base_url"], api_key=cfg["key"])
        t0 = time.time()
        r = c.chat.completions.create(
            model=cfg["model"],
            messages=[{"role": "user", "content": PERGUNTA}],
            temperature=0.0,
        )
        dt = time.time() - t0
        print(f"=== {nome}: {cfg['model']} | {r.usage.total_tokens} tokens | {dt:.1f}s ===")
        print(r.choices[0].message.content.strip(), "\n")
    except Exception as e:
        print(f"=== {nome} FALHOU: {type(e).__name__}: {str(e)[:160]}\n")

print("O modelo maior costuma organizar melhor, e custa mais tokens e mais tempo.")

=== pequeno: llama-3.1-8b-instant | 389 tokens | 0.6s ===
Aqui estão três hipóteses de causa para a vibração anormal na bomba P-204, em ordem de probabilidade:

1. **Desalinhamento ou deslocamento do eixo da bomba**: Isso pode ocorrer devido a uma falha no sistema de suporte ouvido, causando uma vibração excessiva e anormal. A probabilidade de isso ocorrer é alta, pois é uma falha comum em bombas de alta pressão.

2. **Desgaste ou corrosão do rotor ou estator**: O desgaste ou corrosão do rotor ou estator pode causar uma vibração anormal, pois afeta a precisão e a estabilidade da bomba. A probabilidade de isso ocorrer é moderada, pois depende da idade e do uso da bomba.

3. **Problema de balanceamento do rotor**: O problema de balanceamento do rotor pode causar uma vibração anormal, pois afeta a estabilidade da bomba. A probabilidade de isso ocorrer é baixa, pois é uma falha mais complexa e rara em bombas de alta pressão.

É importante notar que essas hipóteses devem ser verificadas e c

## Parte 7 — Os erros que vão acontecer

Você está usando serviço gratuito e compartilhado. Falhar faz parte, e em geral não é culpa sua.

| Código | O que significa | O que fazer |
|---|---|---|
| `401` | chave inválida | gere outra e atualize o Secret |
| `429` | passou do limite por minuto | espere e repita |
| `503` | o modelo está saturado | troque de modelo ou espere |
| `400` | requisição malformada | leia a mensagem: em geral é parâmetro errado |

A célula abaixo provoca um `401` **de propósito**, para você reconhecer a mensagem quando ela aparecer de verdade. No Encontro 3 o seu código passa a tratar `429` e `503` sozinho, com repetição e espera crescente.

In [11]:
try:
    OpenAI(base_url=LLM_BASE_URL, api_key="chave-invalida-de-proposito").chat.completions.create(
        model=LLM_MODEL, messages=[{"role": "user", "content": "oi"}]
    )
except Exception as e:
    print("tipo:", type(e).__name__)
    print("mensagem:", str(e)[:220])
    print("\nEra o esperado. Reconhecer a mensagem economiza tempo depois.")

tipo: AuthenticationError
mensagem: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}

Era o esperado. Reconhecer a mensagem economiza tempo depois.


## Parte 8 — Caminho estendido (opcional)

Para quem terminou antes. Nenhum vale nota; todos voltam mais adiante.

1. **Meça o custo em moeda.** Busque o preço por milhão de tokens do modelo que você usou e calcule quanto custou a Parte 6. Depois estime o custo de rodar 200 vezes por dia, durante um mês.
2. **Force um `429`.** Faça vinte chamadas em sequência e veja o limite aparecer. Anote em qual chamada aconteceu.
3. **Compare instruções de sistema.** Rode a mesma pergunta com três instruções — uma vaga, uma específica, uma com exemplo — e observe o efeito. É o Encontro 4 antecipado.
4. **Reescreva `perguntar` com repetição.** Se der `429` ou `503`, espere 2 s, depois 4 s, depois 8 s. É o Encontro 3 antecipado.
5. **Troque de provedor de verdade.** Pegue um modelo `openai/gpt-oss-*` da lista da Parte 3 e rode a mesma pergunta. Modelo de outro fabricante, mesmo código.

In [ ]:
# Espaco livre para o caminho estendido.


## Parte 9 — Salvar no repositório da equipe

Não há comando a digitar. No menu do Colab:

**Arquivo → Salvar uma cópia no GitHub**

1. Autorize o Colab a acessar sua conta do GitHub — pedido uma única vez
2. Escolha o repositório da sua equipe
3. Caminho do arquivo: `notebooks/enc02_<seu-primeiro-nome>.ipynb`
4. Mensagem do *commit*: `encontro 2: primeira chamada e troca de modelo`
5. **Confirme no navegador** que o arquivo apareceu no repositório

**Um notebook por pessoa, com o seu nome no arquivo.** Assim ninguém sobrescreve o trabalho de ninguém, e o histórico mostra o que cada um fez — o que importa nos marcos, quando cada um defende o próprio trabalho.

O passo a passo completo está no guia **`FLUXO_COLAB_GITHUB.md`**, no repositório da equipe.

### Verificação final

Antes de sair da aula:

- [ ] o notebook rodou de ponta a ponta, sem erro, começando pela célula de preparo
- [ ] a Parte 4 está preenchida e as três perguntas responderam, sem `None`
- [ ] a Parte 6 mostrou os dois modelos respondendo, com tokens e tempo
- [ ] o consumo da Parte 3 está anotado
- [ ] o notebook está no repositório, com o seu nome no arquivo
- [ ] nenhuma chave aparece no código

**Teste que separa quem entendeu:** vá em *Ambiente de execução → Desconectar e excluir ambiente de execução* e rode tudo de novo. Se falhar, alguma célula depende de algo que não está no notebook — e é isso que significa reprodutibilidade.